In [ ]:
!pip install scikit-posthocs

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from google.colab import drive
from scipy.stats import kruskal
import scikit_posthocs as sp
from statsmodels.stats.multitest import multipletests
from scipy.stats import mannwhitneyu, kruskal
from itertools import combinations
from scipy.stats import pearsonr
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df_A1 = pd.read_csv("./estimatedIndividualParameters_A1.txt", sep=',')
df_A2 = pd.read_csv("./estimatedIndividualParameters_A2.txt", sep=',')
df_A3 = pd.read_csv("./estimatedIndividualParameters_A3.txt", sep=',')
df_A4T = pd.read_csv("./estimatedIndividualParameters_A4T.txt", sep=',')
df_A4T.head()

,id,log10beta_SAEM,log10pi_SAEM,log10rho_SAEM,log10phi_SAEM,delta_SAEM,m_SAEM,tau_SAEM,tzero_SAEM,Vzero_SAEM,...,tau_sd,tzero_sd,Vzero_sd,IC50_sd,Study,cat_AgeGroup,cat_LineageBroad,cat_VaccStatus,cat_VaccStatus_0,treatment_cov
0,460_2,-6.67801,2.17551,-3.59447,-5.99992,1.742220,2.21516,12.61590,1.46410,97,...,4.933140,0.153747,0,0.003087,NBA,1,Omicron,3,Boosted,0
1,407_1,-6.50765,2.21887,-3.60290,-5.29483,0.671068,3.19979,7.28455,1.42267,97,...,0.607116,0.167468,0,0.029815,NBA,3,Omicron,3,Boosted,0
2,1053_1,-6.73863,2.27024,-3.58638,-6.09218,0.783146,2.26196,13.13590,1.36113,97,...,8.064300,0.152369,0,0.005097,NBA,1,Omicron,2,Second dose,0
3,2969_1,-6.70599,2.12476,-3.73905,-4.33638,0.691136,3.63305,7.33879,1.41610,97,...,1.644850,0.143234,0,0.005563,NBA,2,Omicron,2,Second dose,0
4,352_1,-6.90343,1.94881,-3.49341,-5.59686,0.740824,3.54585,8.63099,1.42118,97,...,2.320110,0.162128,0,0.002921,NBA,2,Omicron,2,Second dose,0


In [ ]:
models = ["A1","A2","A3", "A4T"]
df = pd.concat([
    df_A1.assign(model="A1"),
    df_A2.assign(model="A2"),
    df_A3.assign(model="A3"),
    df_A4T.assign(model="A4T")
], ignore_index=True)

df["treatment_cov"] = df["treatment_cov"].astype(int)

In [ ]:

df = df[df["Study"]=="Platcov"].copy()

df["group"] = df["model"] + "_" + df["treatment_cov"].astype(str)

group_labels_full = [
    "A1_0","A1_1",
    "A2_0","A2_1",
    "A3_0","A3_1",
    "A4T_0","A4T_1"
]

group_labels_ic50 = [
    "A1_1","A2_1","A3_1", "A4T_1"
]

Viral Dynamic Model Parameter Cross-model comparison (A1, A2, A3, and A4T)

In [ ]:
params = [
    "log10beta_mode",
    "log10pi_mode",
    "log10rho_mode",
    "log10phi_mode",
    "delta_mode",
    "m_mode",
    "tau_mode",
    "tzero_mode"
]

TINY_SIZE = 6
SMALL_SIZE = 8
MEDIUM_SIZE = 10
BIGGER_SIZE = 12

In [ ]:
plt.rc('font', size=SMALL_SIZE)
plt.rc('axes', titlesize=MEDIUM_SIZE)
plt.rc('axes', labelsize=SMALL_SIZE)
plt.rc('xtick', labelsize=TINY_SIZE)
plt.rc('ytick', labelsize=TINY_SIZE)
plt.rc('legend', fontsize=TINY_SIZE)
plt.rc('figure', titlesize=BIGGER_SIZE)


compartment_comparison_results = []

n_cols = 3
ROWS_PER_PARAM = 2
PARAMS_PER_PAGE = 3
MAX_ROWS_PER_PAGE = PARAMS_PER_PAGE * ROWS_PER_PARAM

param_chunks = [params[i:i + PARAMS_PER_PAGE] for i in range(0, len(params), PARAMS_PER_PAGE)]
total_pages = len(param_chunks)

print(f"Total parameters: {len(params)}. Generating a {total_pages}-page PDF with a 6x3 layout...")

with PdfPages("parameter_compartment_analysis_compact.pdf") as pdf:

    for page_idx, chunk in enumerate(param_chunks):
        print(f"\n--- Building Page {page_idx + 1} of {total_pages} ---")

        fig, axes = plt.subplots(nrows=MAX_ROWS_PER_PAGE, ncols=n_cols, figsize=(8.5, 11))
        sns.set_style("ticks")

        if MAX_ROWS_PER_PAGE == 1 and n_cols == 1:
            axes = np.array([[axes]])
        elif MAX_ROWS_PER_PAGE == 1:
            axes = axes[np.newaxis, :]
        elif n_cols == 1:
            axes = axes[:, np.newaxis]

        for param_idx, param in enumerate(chunk):

            current_group_labels = group_labels_ic50 if param == "IC50_mean" else group_labels_full
            df_param = df[df["group"].isin(current_group_labels)].copy()

            if df_param.empty:
                continue

            df_param['Compartment'] = df_param['group'].apply(lambda x: x.split('_')[0])
            pivot_df = df_param.pivot_table(index='id', columns='Compartment', values=param)
            compartments = sorted(pivot_df.columns.tolist())

            if len(compartments) < 2:
                continue

            base_param = param.replace('_mode', '').lower()
            if base_param.startswith('log10'):
                var_name = base_param.replace('log10', '')
                clean_param_name = f"log$_{{10}}$({var_name})"
            else:
                clean_param_name = base_param

            group_pairs = list(combinations(compartments, 2))

            for pair_idx, (comp1, comp2) in enumerate(group_pairs):
                row_idx = (param_idx * ROWS_PER_PARAM) + (pair_idx // n_cols)
                col_idx = pair_idx % n_cols
                ax = axes[row_idx, col_idx]

                paired_data = pivot_df[[comp1, comp2]].dropna()
                d1_clean = paired_data[comp1].values
                d2_clean = paired_data[comp2].values

                if len(d1_clean) < 2:
                    continue

                r_stat, p_val = pearsonr(d1_clean, d2_clean)

                ax.scatter(
                    d1_clean, d2_clean,
                    c='#814185', edgecolors='black',
                    linewidths=0.5, s=15, alpha=0.9, zorder=2
                )

                min_val = min(np.min(d1_clean), np.min(d2_clean))
                max_val = max(np.max(d1_clean), np.max(d2_clean))
                buffer = (max_val - min_val) * 0.05

                ax.plot(
                    [min_val - buffer, max_val + buffer],
                    [min_val - buffer, max_val + buffer],
                    color='#2F5F80', linestyle='-', linewidth=1.0, zorder=1
                )

                stats_text = f"r = {r_stat:.2f}\np = {p_val:.2e}\nn = {len(d1_clean)}"
                ax.text(
                    0.05, 0.95, stats_text,
                    transform=ax.transAxes,
                    fontsize=TINY_SIZE,
                    verticalalignment='top',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='gray', alpha=0.8)
                )

                ax.set_title(f"{clean_param_name}", fontsize=MEDIUM_SIZE, fontweight='bold', pad=4)
                ax.set_xlabel(f"{comp1} Value", fontsize=SMALL_SIZE, labelpad=2)
                ax.set_ylabel(f"{comp2} Value", fontsize=SMALL_SIZE, labelpad=2)

                ax.set_box_aspect(1)

                sns.despine(ax=ax)

                compartment_comparison_results.append({
                    "parameter": param,
                    "comparison": f"{comp1} vs {comp2}",
                    "pearson_r": r_stat,
                    "p_value": p_val,
                    "significant_05": p_val < 0.05,
                    "n_samples": len(d1_clean)
                })


        for r in range(MAX_ROWS_PER_PAGE):
            for c in range(n_cols):
                if not axes[r, c].collections and not axes[r, c].lines:
                    axes[r, c].set_visible(False)

        fig.tight_layout(pad=1.5, w_pad=1.0, h_pad=1.5)

        pdf.savefig(fig, dpi=300)
        plt.close(fig)

print("\nSuccess! Saved to 'parameter_compartment_analysis_compact.pdf'")

Total parameters: 8. Generating a 3-page PDF with a 6x3 layout...

--- Building Page 1 of 3 ---

--- Building Page 2 of 3 ---

--- Building Page 3 of 3 ---

Success! Saved to 'parameter_compartment_analysis_compact.pdf'


VID Parameter Within-model comparison (placebo vs treatment)

In [ ]:
params = [
    "log10beta_mode",
    "log10pi_mode",
    "log10rho_mode",
    "log10phi_mode",
    "delta_mode",
    "m_mode",
    "tau_mode",
    "tzero_mode"
]


TINY_SIZE = 6
SMALL_SIZE = 8
MEDIUM_SIZE = 10
BIGGER_SIZE = 12

plt.rc('font', size=SMALL_SIZE)
plt.rc('axes', titlesize=MEDIUM_SIZE)
plt.rc('axes', labelsize=SMALL_SIZE)
plt.rc('xtick', labelsize=SMALL_SIZE)
plt.rc('ytick', labelsize=TINY_SIZE)
plt.rc('legend', fontsize=TINY_SIZE)
plt.rc('figure', titlesize=BIGGER_SIZE)

compartment_mw_results = []


all_compartments = set()
for param in params:
    labels = group_labels_ic50 if param == "IC50_mean" else group_labels_full
    for lbl in labels:
        all_compartments.add(lbl.split('_')[0])

compartments_global = sorted(list(all_compartments))
n_cols = len(compartments_global)  # This will now automatically be 4

MAX_ROWS_PER_PAGE = 4
param_chunks = [params[i:i + MAX_ROWS_PER_PAGE] for i in range(0, len(params), MAX_ROWS_PER_PAGE)]
total_pages = len(param_chunks)

print(f"Total parameters: {len(params)}. Generating a {total_pages}-page PDF with a 4x4 grid of boxplots...")

with PdfPages("parameter_treatment_analysis_compact.pdf") as pdf:

    for page_idx, chunk in enumerate(param_chunks):
        print(f"\n--- Building Page {page_idx + 1} of {total_pages} ---")

        fig, axes = plt.subplots(nrows=MAX_ROWS_PER_PAGE, ncols=n_cols, figsize=(8.5, 11))
        sns.set_style("ticks")

        if MAX_ROWS_PER_PAGE == 1 and n_cols == 1:
            axes = np.array([[axes]])
        elif MAX_ROWS_PER_PAGE == 1:
            axes = axes[np.newaxis, :]
        elif n_cols == 1:
            axes = axes[:, np.newaxis]


        for row_idx, param in enumerate(chunk):

            current_group_labels = group_labels_ic50 if param == "IC50_mean" else group_labels_full

            base_param = param.replace('_mode', '').lower()
            if base_param.startswith('log10'):
                var_name = base_param.replace('log10', '')
                clean_param_name = f"log$_{{10}}$({var_name})"
            else:
                clean_param_name = base_param

            for col_idx, comp in enumerate(compartments_global):
                ax = axes[row_idx, col_idx]

                group_0 = f"{comp}_0"
                group_1 = f"{comp}_1"

                if group_0 not in current_group_labels or group_1 not in current_group_labels:
                    ax.set_visible(False)
                    continue

                data_0 = df[df["group"] == group_0][param].dropna().to_numpy()
                data_1 = df[df["group"] == group_1][param].dropna().to_numpy()

                if len(data_0) == 0 or len(data_1) == 0:
                    ax.set_visible(False)
                    continue

                # Calculate Mann-Whitney Stats
                if len(np.unique(np.concatenate([data_0, data_1]))) <= 1:
                    stat, p_value = np.nan, np.nan
                    is_significant = False
                    note = "identical values"
                else:
                    stat, p_value = mannwhitneyu(data_0, data_1, alternative='two-sided')
                    is_significant = p_value < 0.05
                    note = ""

                sns.boxplot(
                    data=[data_0, data_1], width=0.5,
                    palette=["#D3D3D3", "#87CEEB"], linewidth=0.8,
                    showfliers=False, ax=ax
                )

                sns.stripplot(
                    data=[data_0, data_1], color="#333333",
                    alpha=0.6, jitter=0.15, size=2.5, ax=ax
                )

                if not np.isnan(stat):
                    stats_text = f"U = {stat:.1f}\np = {p_value:.2e}"
                    ax.text(
                        0.05, 0.95, stats_text,
                        transform=ax.transAxes,
                        fontsize=TINY_SIZE,
                        verticalalignment='top',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='gray', alpha=0.8)
                    )

                ax.set_title(f"{clean_param_name}\n(Comp {comp})", fontsize=MEDIUM_SIZE, fontweight='bold', pad=4)
                ax.set_xticks([0, 1])
                ax.set_xticklabels(["Pbo", "Tx"], fontsize=SMALL_SIZE)

                if col_idx == 0:
                    ax.set_ylabel("Value", fontsize=SMALL_SIZE, labelpad=2)
                else:
                    ax.set_ylabel("")

                ax.set_box_aspect(1)
                sns.despine(ax=ax)
                ax.grid(axis='y', linestyle='--', alpha=0.4)

                compartment_mw_results.append({
                    "parameter": param,
                    "compartment": comp,
                    "group_0": group_0,
                    "group_1": group_1,
                    "U_statistic": stat,
                    "p_value": p_value,
                    "significant": is_significant,
                    "note": note
                })


        for r in range(MAX_ROWS_PER_PAGE):
            for c in range(n_cols):
                if not axes[r, c].collections and not axes[r, c].lines:
                    axes[r, c].set_visible(False)

        fig.tight_layout(pad=1.5, w_pad=1.0, h_pad=1.5)

        pdf.savefig(fig, dpi=300)
        plt.close(fig)

print("\nSuccess! Saved to 'parameter_treatment_analysis_compact.pdf'")

Total parameters: 8. Generating a 2-page PDF with a 4x4 grid of boxplots...

--- Building Page 1 of 2 ---

--- Building Page 2 of 2 ---

Success! Saved to 'parameter_treatment_analysis_compact.pdf'
